# 06 - Owner Notifier

Sends notifications to asset owners about:
- Stale tables requiring attention
- Missing tag compliance
- Overdue certification reviews
- Pending deprecation warnings

Escalates after N days of no response. Logs all notifications for audit.

In [0]:
# Databricks notebook source
import sys as _sys
_nb = (dbutils.notebook.entry_point.getDbutils().notebook()
       .getContext().notebookPath().get())
_sys.path.insert(0, '/Workspace' + '/'.join(_nb.split('/')[:-2]) + '/src')
from lib.common import require_widget
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("control_schema", "uc_hygiene")
dbutils.widgets.text("notification_email", "")
dbutils.widgets.text("slack_webhook_url", "")
dbutils.widgets.text("notification_lookback_days", "7")
dbutils.widgets.text("notification_cooldown_days", "3")
dbutils.widgets.text("notification_batch_limit", "20")


catalog        = require_widget(dbutils, "catalog")
control_schema = require_widget(dbutils, "control_schema")
admin_email    = require_widget(dbutils, "notification_email")
slack_webhook  = dbutils.widgets.get("slack_webhook_url").strip()

notification_lookback_days = int(dbutils.widgets.get("notification_lookback_days") or "7")
notification_cooldown_days = int(dbutils.widgets.get("notification_cooldown_days") or "3")
notification_batch_limit   = int(dbutils.widgets.get("notification_batch_limit")   or "20")
control_fqn    = f"{catalog}.{control_schema}"

print(f"Control schema:   {control_fqn}")
print(f"Admin email:      {admin_email}")
print(f"Slack:            {'configured' if slack_webhook else 'disabled'}")
print(f"Lookback: {notification_lookback_days}d | Cooldown: {notification_cooldown_days}d | Batch limit: {notification_batch_limit}")
import time as _t; _task_start = _t.time()

In [0]:
from datetime import date, timedelta
import uuid

today = date.today()

# Step 1: Gather unresolved findings that need notification
# Only notify once per finding (check notification_log)
pending_notifications = spark.sql(f"""
WITH unresolved AS (
  SELECT 
    sr.scan_id, sr.scan_date, sr.scan_type, 
    sr.catalog_name, sr.schema_name, sr.table_name,
    sr.finding_type, sr.finding_severity, sr.finding_detail,
    sr.recommended_action, sr.owner_email,
    CONCAT(sr.catalog_name, '.', sr.schema_name, '.', sr.table_name) AS asset_ref
  FROM {catalog}.{control_schema}.scan_results sr
  WHERE sr.resolved_at IS NULL
    AND sr.finding_severity IN ('critical', 'warning')
    AND sr.scan_date >= DATE_SUB(CURRENT_DATE(), notification_lookback_days)
),
already_notified AS (
  SELECT DISTINCT asset_reference, notification_type
  FROM {catalog}.{control_schema}.notification_log
  WHERE sent_at >= TIMESTAMP(DATE_SUB(CURRENT_DATE(), notification_cooldown_days))
    AND response_status != 'action_taken'
)
SELECT u.*
FROM unresolved u
LEFT JOIN already_notified an
  ON u.asset_ref = an.asset_reference
  AND u.scan_type = an.notification_type
WHERE an.asset_reference IS NULL
""")

notify_count = pending_notifications.count()
print(f"Pending notifications to send: {notify_count}")

In [0]:
# Step 2: Group by owner and generate notification summaries
if notify_count > 0:
    pending_notifications.createOrReplaceTempView("pending_notifs")

    by_owner = spark.sql(f"""
    SELECT
      COALESCE(owner_email, '{admin_email}') AS recipient,
      COUNT(*) AS finding_count,
      COUNT(CASE WHEN finding_severity = 'critical' THEN 1 END) AS critical_count
    FROM pending_notifs
    GROUP BY COALESCE(owner_email, '{admin_email}')
    """)

    print("\nNotification summary by owner:")
    by_owner.show(truncate=False)

In [0]:
from datetime import datetime, timezone
import uuid

# Step 3: Send notifications
import requests
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

_notif_schema = StructType([
    StructField("notification_id", StringType()),
    StructField("sent_at", TimestampType()),
    StructField("recipient_email", StringType()),
    StructField("notification_type", StringType()),
    StructField("asset_reference", StringType()),
    StructField("message_summary", StringType()),
    StructField("response_status", StringType()),
    StructField("response_at", TimestampType()),
])

def send_slack_notification(webhook_url, message):
    """Send a Slack notification if webhook configured."""
    if not webhook_url:
        return False
    try:
        resp = requests.post(webhook_url, json={"text": message})
        return resp.status_code == 200
    except:
        return False

def format_notification(recipient, findings):
    """Format notification message."""
    lines = [
        f"🧹 **UC Steward Alert** - {today}",
        f"",
        f"The following data assets require your attention:",
        f""
    ]
    for row in findings:
        severity_icon = "🔴" if row.finding_severity == "critical" else "🟡"
        lines.append(f"  {severity_icon} `{row.catalog_name}.{row.schema_name}.{row.table_name}`")
        lines.append(f"     {row.finding_detail}")
        lines.append(f"     Action: {row.recommended_action}")
        lines.append("")

    lines.append("Please resolve these findings or mark assets as needed.")
    lines.append(f"Unresolved critical findings will be auto-deprecated after {notification_lookback_days} day(s) if they continue to age unaddressed.")
    return "\n".join(lines)

notifications_sent = 0
if notify_count > 0:
    owners = by_owner.collect()
    for owner_row in owners:
        recipient = owner_row.recipient
        owner_findings = spark.sql(f"""
        SELECT * FROM pending_notifs
        WHERE COALESCE(owner_email, '{admin_email}') = '{recipient}'
        LIMIT {notification_batch_limit}
        """).collect()

        message = format_notification(recipient, owner_findings)

        # Log the notification via DataFrame to avoid f-string escaping
        notif_id = str(uuid.uuid4())
        asset_refs = ", ".join([f"{r.catalog_name}.{r.schema_name}.{r.table_name}" for r in owner_findings[:5]])
        now = datetime.now(timezone.utc)

        notif_df = spark.createDataFrame(
            [(
                notif_id,
                now,
                recipient,
                "hygiene_alert",
                asset_refs,
                f"{owner_row.finding_count} findings ({owner_row.critical_count} critical)",
                "pending",
                None,
            )],
            _notif_schema,
        )
        notif_df.createOrReplaceTempView("_tmp_notif")
        spark.sql(f"""
        INSERT INTO {catalog}.{control_schema}.notification_log
        SELECT notification_id, sent_at, recipient_email, notification_type,
               asset_reference, message_summary, response_status, response_at
        FROM _tmp_notif
        """)

        if slack_webhook:
            send_slack_notification(slack_webhook, message)

        notifications_sent += 1
        print(f"  📧 Notified: {recipient} ({owner_row.finding_count} findings)")

print(f"""
{'='*52}
  NOTIFICATION RUN COMPLETE
{'='*52}
  Findings queued:  {notify_count}
  Owners notified:  {notifications_sent}
  Slack enabled:    {bool(slack_webhook)}
  Log → {catalog}.{control_schema}.notification_log
{'='*52}
""")

try:
    spark.sql(f"""
    INSERT INTO {catalog}.{control_schema}.job_run_history VALUES (
      CURRENT_DATE(),
      'uc_hygiene_daily_governance',
      'p4_notify_owners',
      'p4_communication',
      'success',
      {notify_count},
      {notifications_sent},
      {notify_count},
      int(_t.time() - _task_start),
      'owners_notified={notifications_sent} slack={'enabled' if slack_webhook else 'disabled'}',
      CURRENT_TIMESTAMP()
    )
    """)
except Exception as _e:
    print(f"Warning: could not write to job_run_history: {_e}")